# 🎨 Generate Figures 1-4 untuk Jurnal TIR BERT-SVR

Notebook ini menghasilkan **4 gambar (Fig 1-4)** untuk jurnal:

| Figure | Deskripsi | Library | Sumber Data |
|--------|-----------|---------|-------------|
| **Fig 1** | Diagram Alur Pipeline Keseluruhan | Graphviz | Desain Arsitektur |
| **Fig 2** | Detail Arsitektur Hybrid BERT-SVR (Horizontal) | Graphviz | Desain Arsitektur |
| **Fig 3** | Kurva Training Loss & F1 per Epoch | Matplotlib | `logs/results_jurnal_final_train_s42.json` |
| Fig 4 | Heatmap Korelasi Pearson 4 Kategori TIR | Seaborn | `prep/y_gold_v3.npy` / `data_ahli_FIXED.csv` |

**Output:** `JURNAL-OUTPUTS/laptop_jurnal/figures/`

---

## ⚙️ Setup Environment & Path (Otomatis Colab/Lokal)

In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from graphviz import Digraph
import warnings
warnings.filterwarnings('ignore')

def setup_paths():
    """Deteksi lingkungan (Colab vs Lokal) dan return dict path."""
    paths = {}
    in_colab = 'google.colab' in sys.modules

    if in_colab:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        DRIVE = '/content/drive/MyDrive'
        print("[INFO] Environment: Google Colab")
    else:
        DRIVE = r'D:/PENELITIAN 1'
        print("[INFO] Environment: Lokal (Windows)")

    paths = {
        'DRIVE': DRIVE,
        'DATA': os.path.join(DRIVE, 'JURNAL-DATASET'),
        'OUT': os.path.join(DRIVE, 'JURNAL-OUTPUTS', 'laptop_jurnal'),
        'PREP': os.path.join(DRIVE, 'JURNAL-OUTPUTS', 'laptop_jurnal', 'prep'),
        'LOG': os.path.join(DRIVE, 'JURNAL-OUTPUTS', 'laptop_jurnal', 'logs'),
        'PRED': os.path.join(DRIVE, 'JURNAL-OUTPUTS', 'laptop_jurnal', 'preds'),
        'FIG': os.path.join(DRIVE, 'JURNAL-OUTPUTS', 'laptop_jurnal', 'figures')
    }

    os.makedirs(paths['FIG'], exist_ok=True)
    print(f"[INFO] Figure output dir: {paths['FIG']}")
    return paths

PATHS = setup_paths()

def save_fig(fig, filename, dpi=300):
    """Simpan figure ke folder FIG."""
    path = os.path.join(PATHS['FIG'], filename)
    fig.savefig(path, dpi=dpi, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    print(f"[SAVED] {path}")

---
## 📊 FIG 1: Diagram Alur Pipeline Keseluruhan (Graphviz)
*Alur: Input → Preprocessing → BERT → Hybrid Fusion → Threshold → Output*

In [ ]:
from graphviz import Digraph

dot = Digraph('Fig1_Pipeline', format='png', graph_attr={
    'rankdir': 'TB',
    'size': '8,10',
    'dpi': '300',
    'fontname': 'DejaVu Sans',
    'bgcolor': 'white',
    'nodesep': '0.4',
    'ranksep': '0.6',
    'splines': 'ortho'
})
dot.attr('node', shape='box', style='filled,rounded', fontname='DejaVu Sans',
         fontsize='11', fillcolor='#e8f0fe', color='#1967d2', fontcolor='#111')
dot.attr('edge', fontname='DejaVu Sans', fontsize='10', color='#333')

# Nodes
dot.node('A', 'INPUT TEKS REFLEKTIF\n(Narasi Quora Bahasa Indonesia)')
dot.node('B', 'PREPROCESSING "KODE UNIK"\n• Normalisasi Emoji 83→7 label [EMO:*]\n• Penanda Tegas [TEGAS]...[/TEGAS]\n• Tokenizer IndoBERT + 9 Special Tokens')
dot.node('C', 'BERT ENCODER (FROZEN)\nindobenchmark/indobert-base-p1\n768-d • 12 Layer • 12 Head')
dot.node('D', 'POOLING GABUNGAN\nCLS + Mean + Max Pooling\n▶ Konkatenasi → 2.304-d Vector')
dot.node('E1', 'BRANCH 1: BERT REGRESI HEAD\n2.304→512(ReLU)→256(ReLU)→4(Sigmoid)\nLoss: Weighted MSE w=[1.4,1.4,1.0,0.9]')
dot.node('E2', 'BRANCH 2: SVR REGRESI\nEmbedding BERT Mentah (Frozen, 768-d)\n+ 8 Fitur Psikometrik → PCA 0.95 (690-d) → SVR RBF (C=10, ε=0.1, γ=scale)')
dot.node('F', 'HYBRID FUSION\nSkor_Final = 0.65×Skor_BERT + 0.35×Skor_SVR\n(Bobot dioptimalkan Val 135)')
dot.node('G', 'THRESHOLD PER-KELAS\nIntrusive:0.30 | Avoidance:0.30\nArousal:0.30 | NegCogMood:0.36')
dot.node('H', 'OUTPUT PREDIKSI\n• 4 Skor Kontinu [0,1]\n• 4 Label Biner (0/1) per Kategori\n• Kategori Dominan + Penjelasan')

# Edges
dot.edges(['AB', 'BC', 'CD'])
dot.edge('D', 'E1')
dot.edge('D', 'E2')
dot.edge('E1', 'F')
dot.edge('E2', 'F', style='dashed', label='SVR Output')
dot.edge('F', 'G')
dot.edge('G', 'H')

# Render
output_path = os.path.join(PATHS['FIG'], 'fig1_pipeline_flow')
dot.render(output_path, cleanup=True)
print(f"[SAVED] Fig 1: {output_path}.png")

---
## 🏗️ FIG 2: Detail Arsitektur Hybrid BERT-SVR (Layout Horizontal Kompak)
*Layout 3-Kolom Horizontal (LR): Input/Encoder | BERT Head+Fusion | SVR Regresi*

In [ ]:
# ==================== FIG 2: ARSITEKTUR DETAIL (VERTIKAL RAPI & LEGA) ====================
dot = Digraph('Fig2_Architecture', format='png', graph_attr={
    'rankdir': 'TB',          # Vertikal: Atas -> Bawah
    'size': '6.5,16',         # Lebar 6.5 inci, Tinggi 16 inci (Lega vertikal)
    'dpi': '300',
    'fontname': 'DejaVu Sans',
    'bgcolor': 'white',
    'nodesep': '0.7',         # Jarak horizontal antar node (Lega)
    'ranksep': '1.5',         # Jarak vertikal antar rank/level (Lega)
    'splines': 'ortho',
    'pad': '0.5',
    'fontname': 'DejaVu Sans'
})
dot.attr('node', shape='box', style='filled,rounded', fontname='DejaVu Sans',
         fontsize='10', fillcolor='#f0f4fa', color='#2c5aa0', fontcolor='#111',
         margin='0.25,0.15',       # Padding dalam node (Lega)
         width='2.8',              # Lebar minimal node (seragam)
         height='0.8')             # Tinggi minimal node
dot.attr('edge', fontname='DejaVu Sans', fontsize='9', color='#444', penwidth='1.2')

# ===== KOLOM 1: INPUT & ENCODER (PALING ATAS) =====
with dot.subgraph(name='cluster_col1') as c:
    c.attr(label=' KOLOM 1: INPUT & ENCODER', style='rounded', color='#1967d2',
           fontname='DejaVu Sans Bold', fontsize='12',
           labelloc='t', labeljust='l')  # Label kiri atas rapi
    # HAPUS rank='same'
    c.node_attr.update(style='filled,rounded', fillcolor='#e8f0fe', color='#1967d2',
                       fontname='DejaVu Sans', fontsize='10')

    c.node('I1', ' INPUT TEKS\nTeks Reflektif Bahasa Indonesia\n(Quora)')
    c.node('P1', ' PREPROCESSING "KODE UNIK"\n• Normalisasi Emoji 83→7 label `[EMO:*]`\n• Penanda Tegas `[TEGAS]...[/TEGAS]`\n• Tokenizer IndoBERT + 9 Special Tokens')
    c.node('B1', ' BERT ENCODER (FROZEN)\n`indobenchmark/indobert-base-p1`\n768-d • 12 Layer • 12 Head')
    c.node('B2', ' POOLING GABUNGAN\nCLS Token (768-d)\nMean Pooling (768-d)\nMax Pooling (768-d)\n▶ Konkatenasi → 2.304-d Vector')

# ===== KOLOM 2: BERT HEAD & FUSION =====
with dot.subgraph(name='cluster_col2') as c:
    c.attr(label=' KOLOM 2: BERT HEAD & HYBRID FUSION', style='rounded', color='#2e7d32',
           fontname='DejaVu Sans Bold', fontsize='12')
    c.node_attr.update(style='filled,rounded', fillcolor='#e8f5e9', color='#2e7d32')

    c.node('H1', ' BERT REGRESI HEAD\n2.304 → 512 (ReLU) → 256 (ReLU) → 4 (Sigmoid)')
    c.node('H2', ' OUTPUT BERT: 4 Skor [0,1]\nIntrusive, Avoidance, Arousal, NegCogMood', _attributes={'style': 'filled,rounded,dashed', 'fillcolor': '#f1f8e9'})
    c.node('H3', ' LOSS: Weighted MSE\nw = [1.4, 1.4, 1.0, 0.9]', _attributes={'style': 'filled,rounded,dashed', 'fillcolor': '#f1f8e9'})

    c.node('F1', ' HYBRID FUSION\nSkor_Final = 0.65×BERT + 0.35×SVR\n(Bobot dioptimalkan Val 135)', _attributes={'fillcolor': '#fff3e0', 'color': '#e65100'})

    c.node('T1', ' THRESHOLD PER-KELAS\nIntrusive:0.30 | Avoidance:0.30\nArousal:0.30 | NegCogMood:0.36')
    c.node('O1', ' OUTPUT FINAL\n• 4 Skor Kontinu [0,1]\n• 4 Label Biner (0/1)\n• Kategori Dominan + Penjelasan')

# ===== KOLOM 3: SVR REGRESI =====
with dot.subgraph(name='cluster_col3') as c:
    c.attr(label=' KOLOM 3: SVR REGRESI (EMBEDDING + PSIKOMETRIK)', style='rounded', color='#f57f17',
           fontname='DejaVu Sans Bold', fontsize='12')
    c.node_attr.update(fillcolor='#fff8e1', color='#f57f17')

    c.node('S1', ' INPUT SVR\n• Embedding BERT Mentah (Frozen, 768-d)\n• 8 Fitur Psikometrik (Kamus Psikolog)')
    c.node('S2', ' PCA 0.95 Varians → 690-d\n(Reduksi Dimensi)', _attributes={'style': 'filled,rounded,dashed', 'fillcolor': '#fffde7'})
    c.node('S3', ' SVR RBF KERNEL\nC=10 | ε=0.1 | γ=scale')
    c.node('S4', ' OUTPUT SVR: 4 Skor [0,1]', _attributes={'style': 'filled,rounded,dashed', 'fillcolor': '#fffde7'})

# ===== EDGES (ALUR UTAMA: ATAS → BAWAH) =====
# Kolom 1 (Atas)
dot.edges([('I1', 'P1'), ('P1', 'B1'), ('B1', 'B2')])

# B2 cabang ke Kolom 2 (BERT Head) & Kolom 3 (SVR Input)
dot.edge('B2', 'H1', label='2.304-d Vector', weight='2')
dot.edge('B2', 'S1', label='Embedding BERT Mentah (Frozen)')

# Kolom 2 Internal (BERT Head → Fusion → Output)
dot.edges([('H1', 'H2'), ('H2', 'H3'), ('H3', 'F1'), ('F1', 'T1'), ('T1', 'O1')])

# Kolom 3 Internal (SVR Pipeline)
dot.edges([('S1', 'S2'), ('S2', 'S3'), ('S3', 'S4')])

# Fusion: SVR Output ke Fusion (dashed)
dot.edge('S4', 'F1', style='dashed', color='#f57f17', label='SVR Score', constraint='true')

# ===== INVISIBLE EDGES UNTUK ALIGNMENT (TIDAK MEMPENGARUHI LAYOUT) =====
dot.edge('B2', 'H1', style='invis', constraint='false')
dot.edge('H3', 'S2', style='invis', constraint='false')
dot.edge('F1', 'S4', style='invis', constraint='false')

# Render
output_path = os.path.join(PATHS['FIG'], 'fig2_architecture_detail')
dot.render(output_path, cleanup=True)
print(f"[SAVED] Fig 2: {output_path}.png")

---
## 📈 FIG 3: Kurva Training Loss & F1 per Epoch (Matplotlib)
*Sumber data: `logs/results_jurnal_final_train_s42.json` (log training asli)*

In [ ]:
# ==================== FIG 3: TRAINING CURVES ====================
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import json
import os

log_path = os.path.join(PATHS['LOG'], 'results_jurnal_final_train_s42.json')

# Cari file log yang ada
if not os.path.exists(log_path):
    possible_logs = [
        os.path.join(PATHS['LOG'], 'results_jurnal_final_train_s42.json'),
        os.path.join(PATHS['LOG'], 'training_history.json'),
    ]
    log_path = next((p for p in possible_logs if os.path.exists(p)), None)

# Default data naskah (fallback)
epochs = list(range(1, 11))
train_loss = [0.0632, 0.0521, 0.0442, 0.0375, 0.0316, 0.0273, 0.0238, 0.0212, 0.0196, 0.0185]
val_loss = [0.0542, 0.0574, 0.0589, 0.0522, 0.0536, 0.0521, 0.0567, 0.0552, 0.0547, 0.0544]
val_f1 = [0.4306, 0.3615, 0.4130, 0.4608, 0.4902, 0.5024, 0.4980, 0.5075, 0.4949, 0.4953]

if log_path and os.path.exists(log_path):
    print(f"[INFO] Load log dari: {log_path}")
    try:
        with open(log_path, 'r') as f:
            data = json.load(f)

        # Coba parse struktur yang umum
        hist = None
        if isinstance(data, list):
            hist = data
        elif isinstance(data, dict):
            if 'history' in data:
                hist = data['history']
            elif 'log_history' in data:  # Transformers Trainer format
                hist = data['log_history']
            elif 'train_loss' in data:  # Dict of lists
                # Asumsi dict of lists: {'epoch': [...], 'train_loss': [...], ...}
                if isinstance(data['train_loss'], list):
                    epochs = data.get('epoch', list(range(1, len(data['train_loss'])+1)))
                    train_loss = data['train_loss']
                    val_loss = data.get('val_loss', data.get('eval_loss', val_loss))
                    val_f1 = data.get('val_f1', data.get('eval_f1', val_f1))
                    hist = None # Sudah diproses

        # Parse hist jika berbentuk list of dicts
        if hist and isinstance(hist, list) and len(hist) > 0 and isinstance(hist[0], dict):
            epochs = [h.get('epoch', h.get('step', i+1)) for i, h in enumerate(hist)]
            train_loss = [h.get('train_loss', h.get('loss', 0)) for h in hist]
            val_loss = [h.get('val_loss', h.get('eval_loss', 0)) for h in hist]
            val_f1 = [h.get('val_f1', h.get('eval_f1', h.get('f1', 0))) for h in hist]
        elif hist and isinstance(hist, dict):
            # Format columnar: {'epoch': [...], 'train_loss': [...]}
            epochs = hist.get('epoch', hist.get('step', epochs))
            train_loss = hist.get('train_loss', hist.get('loss', train_loss))
            val_loss = hist.get('val_loss', hist.get('eval_loss', val_loss))
            val_f1 = hist.get('val_f1', hist.get('eval_f1', hist.get('f1', val_f1)))

        print(f"[INFO] Berhasil parsing log training: {len(epochs)} epoch")

    except Exception as e:
        print(f"[WARN] Gagal parsing log training: {e}. Menggunakan data naskah.")
        # Tetap pakai default data naskah
else:
    print("[WARN] File log training tidak ditemukan. Menggunakan data naskah.")

# ==================== PLOT ====================
fig, ax1 = plt.subplots(figsize=(7, 4.5))

color_loss = '#1f77b4'
color_f1 = '#d62728'

ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('Loss', color=color_loss, fontsize=12)
ax1.plot(epochs, train_loss, 'o-', color=color_loss, label='Train Loss', linewidth=2, markersize=5)
ax1.plot(epochs, val_loss, 's--', color=color_loss, alpha=0.7, label='Val Loss', linewidth=2, markersize=5)
ax1.tick_params(axis='y', labelcolor=color_loss)
ax1.set_ylim(0, max(max(train_loss), max(val_loss)) * 1.2)
ax1.grid(True, linestyle=':', alpha=0.6)

ax2 = ax1.twinx()
color_f1 = '#d62728'
ax2.set_ylabel('Val F1 (Macro/Weighted)', color=color_f1, fontsize=12)
ax2.plot(epochs, val_f1, 'D-', color=color_f1, label='Val F1', linewidth=2, markersize=6)
ax2.tick_params(axis='y', labelcolor=color_f1)
ax2.set_ylim(0, max(val_f1) * 1.15)

# Legend gabung
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='center right', fontsize=10, framealpha=0.9)

plt.title('Kurva Training Loss & Validation F1 per Epoch (Seed 42)', fontsize=13, pad=15, fontweight='bold')
ax1.set_xticks(epochs)
fig.tight_layout()

# Save function (pastikan sudah didefinisikan di cell setup, atau definisikan ulang di sini)
def save_fig(fig, filename, dpi=300):
    path = os.path.join(PATHS['FIG'], filename)
    fig.savefig(path, dpi=dpi, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    print(f"[SAVED] {path}")

save_fig(plt.gcf(), 'fig3_training_curves.png')

---
## 🔥 FIG 4: Heatmap Korelasi Pearson 4 Kategori TIR (Seaborn)
*Sumber data: `prep/y_gold_v3.npy` atau `JURNAL-DATASET/data_ahli_FIXED.csv`*

In [ ]:
# ==================== FIG 4: CORRELATION HEATMAP ====================
y_path = os.path.join(PATHS['PREP'], 'y_gold_v3.npy')
csv_path = os.path.join(PATHS['DATA'], 'data_ahli_FIXED.csv')

categories = ['Intrusive', 'Avoidance', 'Arousal', 'NegCogMood']

if os.path.exists(y_path):
    print(f"[INFO] Load label dari: {y_path}")
    y = np.load(y_path)  # Shape (1981, 4)
    df = pd.DataFrame(y, columns=categories)
elif os.path.exists(csv_path):
    print(f"[INFO] Load label dari CSV: {csv_path}")
    df_raw = pd.read_csv(csv_path)
    label_cols = [c for c in df_raw.columns if c in ['Intrusive', 'Avoidance', 'Arousal', 'NegCogMood']]
    if len(label_cols) == 4:
        df = df_raw[label_cols].astype(float)
        df.columns = categories
    else:
        raise ValueError("Kolom label tidak ditemukan di CSV")
else:
    print("[WARN] File label tidak ditemukan. Menggunakan data simulasi dari naskah.")
    # Data simulasi dari TABEL V naskah
    corr_matrix = np.array([
        [1.000, -0.250, 0.414, 0.320],
        [-0.250, 1.000, -0.119, 0.099],
        [0.414, -0.119, 1.000, 0.344],
        [0.320, 0.099, 0.344, 1.000]
    ])
    df = pd.DataFrame(corr_matrix, index=categories, columns=categories)
    # Plot langsung matrix
    fig, ax = plt.subplots(figsize=(5.5, 4.5))
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
    sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.3f', cmap='RdBu_r',
                center=0, vmin=-0.3, vmax=1, square=True,
                xticklabels=categories, yticklabels=categories,
                cbar_kws={"shrink": 0.8, "label": "Pearson r"}, ax=ax)
    ax.set_title('Heatmap Korelasi Pearson Antar Kategori TIR\n(Gold 1.981)', fontsize=13, pad=15, fontweight='bold')
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    fig.tight_layout()
    save_fig(plt.gcf(), 'fig4_correlation_heatmap.png')
    # Exit early jika pakai simulasi
    import sys
    sys.exit(0)

# Hitung korelasi jika data mentah
corr_matrix = df.corr(method='pearson')

# Plot
fig, ax = plt.subplots(figsize=(5.5, 4.5))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)  # Mask upper triangle
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.3f', cmap='RdBu_r',
            center=0, vmin=-0.3, vmax=1, square=True,
            xticklabels=categories, yticklabels=categories,
            cbar_kws={"shrink": 0.8, "label": "Pearson r"}, ax=ax,
            annot_kws={"size": 11, "weight": "bold"})

ax.set_title('Heatmap Korelasi Pearson Antar Kategori TIR\n(Gold 1.981)', fontsize=13, pad=15, fontweight='bold')
plt.xticks(rotation=45, ha='right', fontsize=11)
plt.yticks(rotation=0, fontsize=11)
fig.tight_layout()

save_fig(plt.gcf(), 'fig4_correlation_heatmap.png')

---
## ✅ Verifikasi Output
Cek file yang dihasilkan di folder `figures`.

In [ ]:
print("=" * 60)
print(f"SEMUA FIGURE (1-4) BERHASIL DISIMPAN DI:")
print(f"{PATHS['FIG']}")
print("=" * 60)
print("File yang dihasilkan:")
for f in ['fig1_pipeline_flow.png', 'fig2_architecture_detail.png',
          'fig3_training_curves.png', 'fig4_correlation_heatmap.png']:
    fp = os.path.join(PATHS['FIG'], f)
    if os.path.exists(fp):
        print(f"  ✅ {f} ({os.path.getsize(fp)/1024:.1f} KB)")
    else:
        print(f"  ❌ {f} (GAGAL)")